# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

> **Data note.** This notebook uses the public 30,000-page starter CSV in the repo (32 clients, 54.2% of pages labelled declining). The capstone uses a different, gated lane cut (10,550 pages, 29 clients, 45.0% declining, separate impression windows), so numbers here are not directly comparable with the capstone paper. Both are observed, directional and for decision-support.

**Two findings from FlyRank's March 2026 research paper, with constructive questions.**

**Finding 1: "365+ day content refreshed within 30 days shows 3.2x health and 57x more impressions."** Where does the outcome come from? Health score is a FlyRank composite that, per the paper, includes visibility inputs, so it is not independent of impressions. Does the design carry the claim? It is a comparison of two groups, not a before/after on the same pages, so pages chosen for refresh could already have had demand. My check on the starter slice (a different cut, so it cannot confirm the 57x): among 365+ day pages, the 0-30 day group has the *lowest* share declining (41.6% against 58.2% for the 91-180 day group, n = 10,484 and 6,967), which fits a refresh benefit, but its median 90-day impressions are lower (390 against 1,625). The refreshed group may simply be different pages. Question: were the compared groups matched on prior demand?

**Finding 2: "Random Forest: average position is the top predictor of health score (43%)."** The paper itself notes that the target is partly built from position and impressions, so a high importance is expected and not a finding about SEO. Question: was the holdout split by brand, or random pages from the same brand? The first can show transfer to new brands, the second tends to overstate it.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/NAJAM2005/flyrank-ml-internship/main/"
PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(PATH if os.path.exists(PATH) else RAW + PATH)      # public starter slice (anonymised)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # outcome for checks only, never a feature
print(df.shape, "| declining share:", round(df["is_declining"].mean(), 3), "| clients:", df["client_id"].nunique())

# Finding 1 in the FlyRank paper: 365+ day pages refreshed within 30 days show health 10.7 -> 34.5 and impressions 71 -> 4,039.
# What can the starter slice say about the comparison it rests on?
old = df[df["age_tier"].astype(str).str.contains("365")]
t = (old.groupby("freshness_tier", observed=True)
        .agg(n=("impressions_90d", "size"), median_impressions_90d=("impressions_90d", "median"),
             decline_rate=("is_declining", "mean")).round(3))
print("365+ day pages by freshness tier (n pages, median 90d impressions, declining share):")
print(t.to_string())
print("\nnote: the starter slice has no health_score column, so the health half of the claim cannot be re-checked here;")
print("the 90-day impressions half can, and it is the one that depends on which pages were refreshed.")

audit = pd.DataFrame([
    {"finding": "Refresh within 30 days: 3.2x health, 57x impressions (365+ day pages)",
     "label / outcome": "health score (composite) and 90-day impressions",
     "does the design carry the claim?": "partly: a group comparison, not a controlled before/after",
     "my question": "Were refreshed pages picked because they already had demand? Is health independent of impressions?"},
    {"finding": "Random Forest: position is the top predictor of health score (43%)",
     "label / outcome": "health score, partly built from position and impressions",
     "does the design carry the claim?": "no for importance: the target contains the inputs (the paper says so)",
     "my question": "Was the holdout split by brand, or random pages from the same brand?"}])
print()
print(audit.to_string(index=False))


(30000, 45) | declining share: 0.542 | clients: 32
365+ day pages by freshness tier (n pages, median 90d impressions, declining share):
                    n  median_impressions_90d  decline_rate
freshness_tier                                             
0-30            10484                   390.0         0.416
181+              174                    15.5         0.471
31-90             103                   613.0         0.612
91-180           6967                  1625.0         0.582

note: the starter slice has no health_score column, so the health half of the claim cannot be re-checked here;
the 90-day impressions half can, and it is the one that depends on which pages were refreshed.

                                                              finding                                          label / outcome                                      does the design carry the claim?                                                                                        my question


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**My model under an honest split (before / after).** Same model and features, only the split changes.

| Split | AUC | Precision in top 10% |
|---|---|---|
| Random pages (before) | 0.725 | 0.836 |
| Grouped by client (after) | 0.622 | 0.599 |

The honest split costs 0.103 AUC and about 24 points of top-10% precision (base rate 54.2%). A random split lets the model see other pages from the same client in training, which is why it looks better than it can be on a new client.

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import roc_auc_score
NUM = ["days_since_last_update", "impressions_90d", "content_age_days", "word_count", "search_volume", "competition", "cpc"]
CAT = ["content_type", "main_intent"]
X = df[NUM].copy()
for c in CAT:
    X[c] = df[c].astype("category")
y, groups = df["is_declining"], df["client_id"]

def run(splitter, grp=None):
    s = pd.Series(np.nan, index=df.index)
    for tr, te in splitter.split(X, y, grp):
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0,
                                           categorical_features="from_dtype").fit(X.iloc[tr], y.iloc[tr])
        s.iloc[te] = m.predict_proba(X.iloc[te])[:, 1]
    return s

def p_top(s, frac=0.10):
    return df.loc[s.sort_values(ascending=False).index[:int(len(df) * frac)], "is_declining"].mean()

rand = run(KFold(5, shuffle=True, random_state=0))
grp = run(GroupKFold(5), groups)
out = pd.DataFrame({"split": ["random pages (before)", "grouped by client (after)"],
                    "AUC": [round(roc_auc_score(y, rand), 3), round(roc_auc_score(y, grp), 3)],
                    "precision_top10%": [round(p_top(rand), 3), round(p_top(grp), 3)]})
print(out.to_string(index=False))
print("\nbase rate:", round(y.mean(), 3))
print("AUC drop from the honest split:", round(roc_auc_score(y, rand) - roc_auc_score(y, grp), 3))


                    split   AUC  precision_top10%
    random pages (before) 0.725             0.836
grouped by client (after) 0.622             0.599

base rate: 0.542
AUC drop from the honest split: 0.103


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit on the final feature set (same hunt as Week 3).**

- No forbidden column (trend fields, health score, last/previous 30-day fields) is among the features.
- No single feature separates the label alone: single-feature AUCs are at most 0.591 (page age), so nothing looks like a hidden copy of the label.
- **Risk found:** `impressions_90d` contains the label window (it is at least last-30 plus previous-30 in 100% of rows). Without it the grouped AUC drops from 0.622 to **0.560**. So the starter-slice model's strength partly depends on a column that includes the window being predicted, and 0.62 should be read as an upper bound. The capstone avoids this by using separate windows.

In [3]:
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining", "health_score",
             "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
             "sessions_last_30d", "sessions_prev_30d"]
FEATURES = NUM + CAT
print("forbidden columns among features:", [c for c in FEATURES if c in FORBIDDEN] or "none")

# 1) single-feature AUC: a feature that alone separates the label strongly is a leakage suspect
rows = []
for c in NUM:
    v = df[[c, "is_declining"]].dropna()
    a = roc_auc_score(v["is_declining"], v[c])
    rows.append({"feature": c, "n": len(v), "single_feature_AUC": round(max(a, 1 - a), 3)})
print()
print(pd.DataFrame(rows).sort_values("single_feature_AUC", ascending=False).to_string(index=False))

# 2) containment: impressions_90d contains the label window (last 30d), so it is a risk
c = (df["impressions_90d"] >= df["impressions_last_30d"] + df["impressions_prev_30d"]).mean()
print(f"\nimpressions_90d >= last30 + prev30 in {c:.1%} of rows -> the 90-day total includes the label window")

# 3) sensitivity: drop impressions_90d and see how much the honest result moves
cols_wo = [c for c in X.columns if c != "impressions_90d"]
s2 = pd.Series(np.nan, index=df.index)
for tr, te in GroupKFold(5).split(X, y, groups):
    m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0,
                                       categorical_features="from_dtype").fit(X.iloc[tr][cols_wo], y.iloc[tr])
    s2.iloc[te] = m.predict_proba(X.iloc[te][cols_wo])[:, 1]
print("grouped AUC with impressions_90d   :", round(roc_auc_score(y, grp), 3))
print("grouped AUC without impressions_90d:", round(roc_auc_score(y, s2), 3))


forbidden columns among features: none

               feature     n  single_feature_AUC
      content_age_days 30000               0.591
       impressions_90d 30000               0.584
         search_volume 27532               0.561
            word_count 22301               0.546
days_since_last_update 30000               0.527
           competition 27532               0.519
                   cpc 27532               0.513

impressions_90d >= last30 + prev30 in 100.0% of rows -> the 90-day total includes the label window


grouped AUC with impressions_90d   : 0.622
grouped AUC without impressions_90d: 0.56


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Claim rewrite.** My boldest sentence was "The model predicts which pages will decline." The safe version, using the numbers above: *"On pages from clients held out of training, the model ranked declining pages above others with an AUC of about 0.62 (0.5 is random), and about 0.56 without the one column that includes the label window; this is a weak, directional signal for triage, not a prediction of individual pages."* All figures are observed, from one snapshot, and for decision-support.

In [4]:
auc_after = roc_auc_score(y, grp)
claims = pd.DataFrame([
    {"version": "bold (before)", "sentence": "The model predicts which pages will decline."},
    {"version": "safe (after)",
     "sentence": f"On pages from clients held out of training, the model ranked declining pages above others "
                 f"with an AUC of about {auc_after:.2f} (0.5 is random); this is a weak, directional signal for "
                 f"triage and not a prediction of individual pages."}])
pd.set_option("display.max_colwidth", 200)
print(claims.to_string(index=False))


      version                                                                                                                                                                                                                     sentence
bold (before)                                                                                                                                                                                 The model predicts which pages will decline.
 safe (after) On pages from clients held out of training, the model ranked declining pages above others with an AUC of about 0.62 (0.5 is random); this is a weak, directional signal for triage and not a prediction of individual pages.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.